In [45]:
import asyncio
import os
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def main():
    serve_dir = "../mcp-demo-files"
    print(f"[DEBUG] Serving directory: {serve_dir}")
    print(f"[DEBUG] Directory exists: {os.path.isdir(serve_dir)}")
    
    server_params = StdioServerParameters(
        command="npx",
        args=[
            "-y",
            "@modelcontextprotocol/server-filesystem",
            serve_dir
        ],
    )

    try:
        print("[DEBUG] Starting MCP server connection...")
        async with stdio_client(server_params) as (read, write):
            print("[DEBUG] Connected! Initializing session...")
            async with ClientSession(read, write) as session:
                print("[DEBUG] Initializing...")
                await session.initialize()
                print("[DEBUG] ✓ Session initialized!")

                print("\n[DEBUG] Listing tools...")
                tools = await session.list_tools()
                print(f"[DEBUG] Got {len(tools.tools)} tools")

                print("\nAvailable tools:")
                for tool in tools.tools:
                    print(f"  - {tool.name}: {tool.description}")

                print("\n[DEBUG] Calling list_directory...")
                result = await session.call_tool(
                    "list_directory",
                    arguments={"path": "."}
                )

                print("\nTool result:")
                # print(result)
                # Format the result nicely for <class 'mcp.types.CallToolResult'>
                # print(f"Type of Result: {type(result)}")
                # print(result.content)
                # print(result.content[0])
                # text = result.content[0]
                # print(type(text))

                file_list_text = result.content[0].text
                # print(f"{file_list_text}")

                lines = file_list_text.splitlines()
                for line in lines:
                    print(f"  - {line}")

    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

# if __name__ == "__main__":
#     asyncio.run(main())

In [46]:
await main()

[DEBUG] Serving directory: ../mcp-demo-files
[DEBUG] Directory exists: True
[DEBUG] Starting MCP server connection...
[DEBUG] Connected! Initializing session...
[DEBUG] Initializing...
[DEBUG] ✓ Session initialized!

[DEBUG] Listing tools...
[DEBUG] Got 14 tools

Available tools:
  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
  - read_text_file: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
  - read_media_file: Read an image or audio file. Returns the base64 encoded data and MIME type. Only works within allowed directories.
  - 